# Silver — Limpieza y transformación
**Proyecto:** AI Tech Landscape Pipeline  
**Descripción:** Limpia, estandariza tipos y prepara los 3 datasets para la capa Gold.

In [ ]:
import pandas as pd
import re
import os

SILVER_PATH = "../data/silver/"
GOLD_PATH = "../data/gold/"
os.makedirs(GOLD_PATH, exist_ok=True)

df_stocks = pd.read_parquet(f"{SILVER_PATH}raw_big_tech_stocks.parquet")
df_gpu    = pd.read_parquet(f"{SILVER_PATH}raw_gpu_companies.parquet")
df_ai     = pd.read_parquet(f"{SILVER_PATH}raw_ai_companies.parquet")

print("✓ Datasets cargados")

## Big Tech Stocks — limpieza

In [ ]:
df_stocks["Date"] = pd.to_datetime(df_stocks["Date"])
df_stocks.columns = df_stocks.columns.str.strip().str.lower().str.replace(" ", "_")
df_stocks = df_stocks.drop(columns=["source_file"])

nulos = df_stocks.isnull().sum().sum()
df_stocks = df_stocks.dropna()

print(f"Nulos eliminados : {nulos}")
print(f"Rango de fechas  : {df_stocks['date'].min()} → {df_stocks['date'].max()}")
print(f"Empresas         : {sorted(df_stocks['ticker'].unique())}")
print(f"Filas finales    : {len(df_stocks):,}")
df_stocks.head(3)

## GPU Companies — limpieza

In [ ]:
df_gpu.columns = df_gpu.columns.str.strip().str.lower().str.replace(" ", "_")
df_gpu["date"] = pd.to_datetime(df_gpu["date"])
df_gpu = df_gpu.drop(columns=["source_file"])
df_gpu = df_gpu.dropna()
df_gpu["empresa"] = df_gpu["empresa"].str.upper().str.strip()

print(f"Empresas GPU     : {sorted(df_gpu['empresa'].unique())}")
print(f"Rango de fechas  : {df_gpu['date'].min()} → {df_gpu['date'].max()}")
print(f"Filas finales    : {len(df_gpu):,}")
df_gpu.head(3)

## AI Companies — limpieza

In [ ]:
df_ai.columns = df_ai.columns.str.strip().str.lower().str.replace(" ", "_")

def limpiar_revenue(valor):
    if pd.isna(valor):
        return None
    valor = str(valor).replace("$", "").replace(",", "").strip()
    if "billion" in valor.lower():
        return float(re.sub(r"[^\d.]", "", valor)) * 1_000_000_000
    elif "million" in valor.lower():
        return float(re.sub(r"[^\d.]", "", valor)) * 1_000_000
    else:
        try:
            return float(re.sub(r"[^\d.]", "", valor))
        except:
            return None

df_ai["revenue_usd"] = df_ai["annual_revenue"].apply(limpiar_revenue)
df_ai["glassdoor_score"] = df_ai["glassdoor_score"].str.replace("/5", "").str.strip().astype(float)
df_ai["founded"] = pd.to_numeric(df_ai["founded"], errors="coerce").astype("Int64")

print(f"Empresas AI      : {len(df_ai)}")
print(f"Revenue parseado : {df_ai['revenue_usd'].notna().sum()} de {len(df_ai)}")
print(f"Nulos en revenue : {df_ai['revenue_usd'].isna().sum()}")
df_ai[["company", "founded", "annual_revenue", "revenue_usd", "glassdoor_score"]].head(10)

## Guardar Silver

In [ ]:
df_stocks.to_parquet(f"{SILVER_PATH}silver_big_tech_stocks.parquet", index=False)
df_gpu.to_parquet(f"{SILVER_PATH}silver_gpu_companies.parquet",      index=False)
df_ai.to_parquet(f"{SILVER_PATH}silver_ai_companies.parquet",        index=False)

print("=" * 50)
print("SILVER COMPLETADO")
print("=" * 50)
print(f"  Big Tech Stocks : {len(df_stocks):,} filas — {df_stocks['ticker'].nunique()} empresas")
print(f"  GPU Companies   : {len(df_gpu):,} filas — {df_gpu['empresa'].nunique()} empresas")
print(f"  AI Companies    : {len(df_ai):,} filas")
print("\nDatos listos para capa Gold →")